# **Import**

In [ ]:
# Install required packages
%pip install -q datasets
%pip install -q transformers
%pip install -q sentence-transformers
%pip install -q nltk


import os
import sys
from datasets import load_dataset
import json
from transformers import pipeline, set_seed
import random
import torch
from huggingface_hub import login
import getpass
import gc

import string
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.translate import meteor_score

import pandas as pd
from IPython.display import display

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


# **Seed to ensure reproducibility**

In [2]:
def set_seeds(seed=42):

    random.seed(seed)
    # In caso dovessimo usare Numpy
    # np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    # Seed on transformers library
    set_seed(seed)


set_seeds(seed=42)

# **Hugging Face Login**

In [3]:
# Insert the hugging face token
print("Please enter your Hugging Face Access Token: (You must have access to Llama 3.2):")
hf_token = getpass.getpass()
login(token=hf_token)

Please enter your Hugging Face Access Token: (You must have access to Llama 3.2):


# **Environment Setup**

In [30]:
colab = False
kaggle = False

# Automatic Environment Detection
if 'google.colab' in sys.modules:
    print("🌐 Environment detected: Google Colab")
    colab = True

    # Automatically mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    BASE_DIR = '/content/drive/MyDrive/MNLP_2026_HW2_2278125_2283937/'

elif 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    print("🔵 Environment detected: Kaggle")
    kaggle = True
    
    BASE_DIR = '/kaggle/working/'

else:
    print("💻 Environment detected: Local Machine")
    BASE_DIR = 'C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026'

# Defining paths for subdirectories
JSON_DIR = os.path.join(BASE_DIR, 'JSON')
PLOTS_DIR = os.path.join(BASE_DIR, 'plots')
OUTPUTS_DIR = os.path.join(BASE_DIR, 'outputs')
NO_REQUESTED_OUTPUTS_DIR = os.path.join(BASE_DIR, 'no_requested_outputs')

# Creating directories if they don't exist
os.makedirs(JSON_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(NO_REQUESTED_OUTPUTS_DIR, exist_ok=True)

print(f"📁 Base Directory set to: {BASE_DIR}")

# Set up hardware acceleration (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device selected: {device}")

💻 Environment detected: Local Machine
📁 Base Directory set to: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026
🚀 Device selected: cuda


# **GPU Usage Debugging Function**

In [5]:
def gpu_status(label=""):
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved  = torch.cuda.memory_reserved(0)  / 1024**3
    total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free      = total - reserved
    print(f"[GPU {label}]")
    print(f"  Allocated:  {allocated:.2f} GB")
    print(f"  Reserved: {reserved:.2f} GB")
    print(f"  Free:    {free:.2f} GB")
    print(f"  Total:    {total:.2f} GB")

# **Dataset Loading and Exploration**

In [6]:
ds = load_dataset(
    "sapienzanlp-course-materials/hw-mnlp-2026"
)

# Sanity check
for split in ds.keys():
    print(f"Split: {split:6} | Features: {ds[split].column_names}")

# Unpack splits into dedicated variables for easier access
train = ds['train']
test = ds['test']
blind = ds['blind']

Split: test   | Features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer']
Split: blind  | Features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer']
Split: train  | Features: ['wikipedia_title', 'wikidata_id', 'query', 'query_id', 'candidate_chunks', 'n_candidates', 'answer', 'answer_pos', 'short_answer']


# **Rankings Loading**

In [7]:
# Rankings file name definition
rankings_test_filename = 'hit_k_1-test-distilbert-triplet-cosine_(reranking).jsonl'
rankings_blind_filename = 'hit_k_1-blind-distilbert-triplet-cosine_(reranking).jsonl'

# Complete path creation
rankings_test_path = os.path.join(JSON_DIR, rankings_test_filename)
rankings_blind_path = os.path.join(JSON_DIR, rankings_blind_filename)

retrieved_results_test = []
retrieved_results_blind = []

# Test set Rankings loading
try:
    with open(rankings_test_path, 'r', encoding='utf-8') as f:
        for line in f:
            retrieved_results_test.append(json.loads(line.strip()))

    print(f"✅ Test Set rankings loaded with success!")
    print(f"   📍 File: {rankings_test_path}")

except FileNotFoundError:
    print(f"❌ ERROR: TEST SET rankings does not found:\n{rankings_test_path}\n")


# Blind set Rankings loading
try:
    with open(rankings_blind_path, 'r', encoding='utf-8') as f:
        for line in f:
            retrieved_results_blind.append(json.loads(line.strip()))

    print(f"✅ Blind Set rankings loaded with success!")
    print(f"   📍 File: {rankings_blind_path}")

except FileNotFoundError:
    print(f"❌ ERROR: BLIND SET rankings does not found:\n{rankings_blind_path}")

# Unique Dictionary creation {'query_id1': [rankings], 'query_id2': [rankings], ...}
rankings_test = {}
for item in retrieved_results_test:
    rankings_test.update(item)

rankings_blind = {}
for item in retrieved_results_blind:
    rankings_blind.update(item)

✅ Test Set rankings loaded with success!
   📍 File: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\JSON\hit_k_1-test-distilbert-triplet-cosine_(reranking).jsonl
✅ Blind Set rankings loaded with success!
   📍 File: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\JSON\hit_k_1-blind-distilbert-triplet-cosine_(reranking).jsonl


# **Small LMs Implementation**

In [ ]:
# Function to generate prompts in RAG and Oracle settings for both test and blind sets
def prompts_generation(dataset, rankings, k, oracle=False):

        all_prompts = []
        all_retrieved_chunks = []

        queries = dataset['query']
        query_ids = dataset['query_id']
        candidate_chunks = dataset['candidate_chunks']

        blind = all(pos == 100 for pos in dataset['answer_pos'])

        # Mapping query_ids to their corresponding answer positions (only for non-blind datasets)
        if not blind:
            mapped_answers = dict(zip(query_ids, dataset['answer_pos']))
        else:
            mapped_answers = {}
            if oracle:
                print("⚠️ Note: Blind Dataset detected (all values are 100). Oracle mode will be ignored to avoid errors.")

        # Generating prompts for each query
        for q_id, q_text, candidates in zip(query_ids, queries, candidate_chunks):

            # Retrieving the top-k chunk ids for the current query
            best_chunk_positions = rankings.get(q_id, [])[:k]

            context_block = ""

            # We iterate over the top-k chunk positions to build the context block for the prompt
            for i, c_position in enumerate(best_chunk_positions, 1):

                # If oracle is True, we replace the top-ranked chunk with the answer
                if i==1 and oracle and q_id in mapped_answers:
                    answer_pos = mapped_answers[q_id]
                    best_chunk_positions[0] = answer_pos
                    chunk_text = candidates[answer_pos]
                else:
                    chunk_text = candidates[c_position]

                context_block += f"{i}. {chunk_text}\n"

            # Writing the prompt with the retrieved context and the query
            full_prompt = (
            f"Given the following information:\n"
            f"{context_block}"
            f"Reply to this question:\n"
            f"{q_text}?"
            )

            all_prompts.append(full_prompt)
            all_retrieved_chunks.append(best_chunk_positions)

        return all_prompts, all_retrieved_chunks

# Function to generate answers for a given dataset and generator
def answers(dataset, generator, settings='Baseline', k=3):

    answers = {}

    split = str(dataset.split).capitalize()

    # We get all the queries to work with batches
    queries = list(dataset['query'])
    query_ids = list(dataset['query_id'])

    # Depending on the settings, we either use the raw queries (Baseline) or generate prompts with retrieved context (RAG and Oracle)
    if settings == 'Baseline':
        prompts = queries
        retrieved_chunks = [[] for _ in range(len(queries))]
    elif settings in ['RAG', 'Oracle']:
        prompts, retrieved_chunks = prompts_generation(dataset, rankings_test if split=='Test' else rankings_blind, k=k, oracle=(settings=='Oracle'))
    else:
        raise ValueError(f"Invalid settings: {settings}. Choose from 'Baseline', 'RAG', or 'Oracle'.")


    # Batch generation of answers using the provided generator pipeline
    print(f"🔍 Starting generation for {len(queries)} queries in {split} set with {settings} setting...")

    # Set the batch size dynamically: 8 on Colab, 16 locally
    current_batch_size = 8 if colab else 16

    outputs = generator(
        prompts,
        batch_size=current_batch_size,
        max_new_tokens=50,
        truncation=True,
        return_full_text=False,
        do_sample=False,
        generation_config=None,
        pad_token_id=generator.tokenizer.eos_token_id
    )
    print(f"✅ Generation completed for {len(queries)} queries in {split} set with {settings} setting.")

    # We iterate over the generated outputs and store them in the answers dictionary
    for q_id, out, prompt, chunks in zip(query_ids, outputs, prompts, retrieved_chunks):

        generated_answer = out[0]['generated_text'].strip()

        answers[q_id] = {
            "query_id": q_id,
            "retrieved_chunks": chunks,
            "augmented_prompt": prompt if settings in ["RAG", "Oracle"] else None,
            "generated_answer": generated_answer
        }

    return answers

## **Llama-3.2-3B-Instruct**

In [9]:
llama_id = "meta-llama/Llama-3.2-3B-Instruct"
llama = pipeline(
    "text-generation",
    model=llama_id,
    dtype=torch.bfloat16,
    device_map="auto",
    token = True
)

llama.tokenizer.pad_token = llama.tokenizer.eos_token
llama.tokenizer.padding_side = 'left'
llama.tokenizer.model_max_length = 512
llama.generation_config.max_length = None

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

### **Baseline**

In [10]:
Llama_baseline_test_answers = answers(test, generator=llama)
print("✅ Llama Baseline Test Answers Generation Completed!")
Llama_baseline_blind_answers = answers(blind, generator=llama)
print("✅ Llama Baseline Blind Answers Generation Completed!")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


🔍 Starting generation for 2000 queries in Test set with Baseline setting...
✅ Generation completed for 2000 queries in Test set with Baseline setting.
✅ Llama Baseline Test Answers Generation Completed!
🔍 Starting generation for 1322 queries in Blind set with Baseline setting...
✅ Generation completed for 1322 queries in Blind set with Baseline setting.
✅ Llama Baseline Blind Answers Generation Completed!


### **RAG**

In [11]:
Llama_RAG_test_answers = answers(test, generator=llama, settings='RAG', k=3)
print("✅ Llama RAG Test Answers Generation Completed!")
Llama_RAG_blind_answers = answers(blind, generator=llama, settings='RAG', k=3)
print("✅ Llama RAG Blind Answers Generation Completed!")

🔍 Starting generation for 2000 queries in Test set with RAG setting...
✅ Generation completed for 2000 queries in Test set with RAG setting.
✅ Llama RAG Test Answers Generation Completed!
🔍 Starting generation for 1322 queries in Blind set with RAG setting...
✅ Generation completed for 1322 queries in Blind set with RAG setting.
✅ Llama RAG Blind Answers Generation Completed!


### **Oracle**

In [12]:
Llama_Oracle_test_answers = answers(test, generator=llama, settings='Oracle', k=3)
print("✅ Llama Oracle Test Answers Generation Completed!")
Llama_Oracle_blind_answers = answers(blind, generator=llama, settings='Oracle', k=3)
print("✅ Llama Oracle Blind Answers Generation Completed!")

🔍 Starting generation for 2000 queries in Test set with Oracle setting...
✅ Generation completed for 2000 queries in Test set with Oracle setting.
✅ Llama Oracle Test Answers Generation Completed!
⚠️ Note: Blind Dataset detected (all values are 100). Oracle mode will be ignored to avoid errors.
🔍 Starting generation for 1322 queries in Blind set with Oracle setting...
✅ Generation completed for 1322 queries in Blind set with Oracle setting.
✅ Llama Oracle Blind Answers Generation Completed!


## **Qwen3-0.6B**

In [13]:
# We free up GPU memory by deleting the Llama pipeline and clearing the cache before initializing Qwen
if 'llama' in locals():
    del llama

gc.collect()
torch.cuda.empty_cache()

qwen_id = "Qwen/Qwen3-0.6B"
qwen = pipeline(
    "text-generation",
    model=qwen_id,
    dtype=torch.bfloat16,
    device_map="auto",
    token = True
)

qwen.generation_config.max_length = None

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

### **Baseline**

In [14]:
Qwen_baseline_test_answers = answers(test, generator=qwen)
print("✅ Qwen Baseline Test Answers Generation Completed!")
Qwen_baseline_blind_answers = answers(blind, generator=qwen)
print("✅ Qwen Baseline Blind Answers Generation Completed!")

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🔍 Starting generation for 2000 queries in Test set with Baseline setting...
✅ Generation completed for 2000 queries in Test set with Baseline setting.
✅ Qwen Baseline Test Answers Generation Completed!
🔍 Starting generation for 1322 queries in Blind set with Baseline setting...
✅ Generation completed for 1322 queries in Blind set with Baseline setting.
✅ Qwen Baseline Blind Answers Generation Completed!


### **RAG**

In [15]:
Qwen_RAG_test_answers = answers(test, generator=qwen, settings='RAG', k=3)
print("✅ Qwen RAG Test Answers Generation Completed!")
Qwen_RAG_blind_answers = answers(blind, generator=qwen, settings='RAG', k=3)
print("✅ Qwen RAG Blind Answers Generation Completed!")

🔍 Starting generation for 2000 queries in Test set with RAG setting...
✅ Generation completed for 2000 queries in Test set with RAG setting.
✅ Qwen RAG Test Answers Generation Completed!
🔍 Starting generation for 1322 queries in Blind set with RAG setting...
✅ Generation completed for 1322 queries in Blind set with RAG setting.
✅ Qwen RAG Blind Answers Generation Completed!


### **Oracle**

In [16]:
Qwen_Oracle_test_answers = answers(test, generator=qwen, settings='Oracle', k=3)
print("✅ Qwen Oracle Test Answers Generation Completed!")
Qwen_Oracle_blind_answers = answers(blind, generator=qwen, settings='Oracle', k=3)
print("✅ Qwen Oracle Blind Answers Generation Completed!")

🔍 Starting generation for 2000 queries in Test set with Oracle setting...
✅ Generation completed for 2000 queries in Test set with Oracle setting.
✅ Qwen Oracle Test Answers Generation Completed!
⚠️ Note: Blind Dataset detected (all values are 100). Oracle mode will be ignored to avoid errors.
🔍 Starting generation for 1322 queries in Blind set with Oracle setting...
✅ Generation completed for 1322 queries in Blind set with Oracle setting.
✅ Qwen Oracle Blind Answers Generation Completed!


## **Output Files**

In [31]:
def save_jsonl(answers_dict, filename, dir):
    full_path = os.path.join(dir, filename)
    with open(full_path, 'w', encoding='utf-8') as f:
        for q_id in answers_dict:
            json.dump(answers_dict[q_id], f, ensure_ascii=False)
            f.write('\n')
    print(f"💾 File saved: {full_path}")

# ==========================================
# Llama-3.2-3B-Instruct
# ==========================================
# We save the Baseline outputs in a separate directory since they are not requested for submission

# Baseline Output Files
save_jsonl(Llama_baseline_test_answers, "hit_k_1_-all-test-Llama-3.2-3B-Instruct-baseline.jsonl", NO_REQUESTED_OUTPUTS_DIR)
save_jsonl(Llama_baseline_blind_answers, "hit_k_1_-all-blind-Llama-3.2-3B-Instruct-baseline.jsonl", NO_REQUESTED_OUTPUTS_DIR)

# RAG Output Files
save_jsonl(Llama_RAG_test_answers, "hit_k_1_-all-test-Llama-3.2-3B-Instruct.jsonl", OUTPUTS_DIR)
save_jsonl(Llama_RAG_blind_answers, "hit_k_1_-all-blind-Llama-3.2-3B-Instruct.jsonl", OUTPUTS_DIR)

# Oracle Output Files
save_jsonl(Llama_Oracle_test_answers, "hit_k_1_-all-test-Llama-3.2-3B-Instruct-oracle.jsonl", NO_REQUESTED_OUTPUTS_DIR)
save_jsonl(Llama_Oracle_blind_answers, "hit_k_1_-all-blind-Llama-3.2-3B-Instruct-oracle.jsonl", NO_REQUESTED_OUTPUTS_DIR)

# ==========================================
# Qwen3-0.6B
# ==========================================

# Baseline Output Files
save_jsonl(Qwen_baseline_test_answers, "hit_k_1_-all-test-Qwen3-0.6B-baseline.jsonl", NO_REQUESTED_OUTPUTS_DIR)
save_jsonl(Qwen_baseline_blind_answers, "hit_k_1_-all-blind-Qwen3-0.6B-baseline.jsonl", NO_REQUESTED_OUTPUTS_DIR)   

# RAG Output Files
save_jsonl(Qwen_RAG_test_answers, "hit_k_1_-all-test-Qwen3-0.6B.jsonl", OUTPUTS_DIR)
save_jsonl(Qwen_RAG_blind_answers, "hit_k_1_-all-blind-Qwen3-0.6B.jsonl", OUTPUTS_DIR)

# Oracle Output Files
save_jsonl(Qwen_Oracle_test_answers, "hit_k_1_-all-test-Qwen3-0.6B-oracle.jsonl", NO_REQUESTED_OUTPUTS_DIR)
save_jsonl(Qwen_Oracle_blind_answers, "hit_k_1_-all-blind-Qwen3-0.6B-oracle.jsonl", NO_REQUESTED_OUTPUTS_DIR)

💾 File saved: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\no_requested_outputs\hit_k_1_-all-test-Llama-3.2-3B-Instruct-baseline.jsonl
💾 File saved: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\no_requested_outputs\hit_k_1_-all-blind-Llama-3.2-3B-Instruct-baseline.jsonl
💾 File saved: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\outputs\hit_k_1_-all-test-Llama-3.2-3B-Instruct.jsonl
💾 File saved: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\outputs\hit_k_1_-all-blind-Llama-3.2-3B-Instruct.jsonl
💾 File saved: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\no_requested_

# **Automatic Evaluation**

## **JSON Loading**

In [32]:
# We re load the predictions just to be free of executing the cell below without running the entire notebook 

# This function takes as input a JSON file and reload the exact same dictionary as before
def load_jsonl(filename, dir):
   
    full_path = os.path.join(dir, filename)
    loaded_answers = {}
    
    try:
        with open(full_path, 'r', encoding='utf-8') as f:
            for line in f:
                # Skip empty lines
                if not line.strip():
                    continue
                
                # Converts the JSON string back to a Python dictionary
                record = json.loads(line)
                
                # Extract the query_id to use as key in the loaded_answers dictionary
                q_id = record['query_id']
                loaded_answers[q_id] = record
                
        print(f"✅ File loaded successfully: {full_path} ({len(loaded_answers)} query)")
        return loaded_answers
        
    except FileNotFoundError:
        print(f"❌ Error: The file '{full_path}' was not found.")
        return {}
    except Exception as e:
        print(f"❌ Error during the reading of '{full_path}': {e}")
        return {}
    
# ==========================================
# Reloading LLama 3.2 3B
# ==========================================
print("Loading Llama results...")

Llama_baseline_test_answers = load_jsonl("hit_k_1_-all-test-Llama-3.2-3B-Instruct-baseline.jsonl", NO_REQUESTED_OUTPUTS_DIR)
Llama_baseline_blind_answers = load_jsonl("hit_k_1_-all-blind-Llama-3.2-3B-Instruct-baseline.jsonl", NO_REQUESTED_OUTPUTS_DIR)

Llama_RAG_test_answers = load_jsonl("hit_k_1_-all-test-Llama-3.2-3B-Instruct.jsonl", OUTPUTS_DIR)
Llama_RAG_blind_answers = load_jsonl("hit_k_1_-all-blind-Llama-3.2-3B-Instruct.jsonl", OUTPUTS_DIR)

Llama_Oracle_test_answers = load_jsonl("hit_k_1_-all-test-Llama-3.2-3B-Instruct-oracle.jsonl", NO_REQUESTED_OUTPUTS_DIR)
Llama_Oracle_blind_answers = load_jsonl("hit_k_1_-all-blind-Llama-3.2-3B-Instruct-oracle.jsonl", NO_REQUESTED_OUTPUTS_DIR)


# ==========================================
# Reloading Qwen 0.6B (o 2.5)
# ==========================================
print("\nLoading Qwen results...")

Qwen_baseline_test_answers = load_jsonl("hit_k_1_-all-test-Qwen3-0.6B-baseline.jsonl", NO_REQUESTED_OUTPUTS_DIR)
Qwen_baseline_blind_answers = load_jsonl("hit_k_1_-all-blind-Qwen3-0.6B-baseline.jsonl", NO_REQUESTED_OUTPUTS_DIR)

Qwen_RAG_test_answers = load_jsonl("hit_k_1_-all-test-Qwen3-0.6B.jsonl", OUTPUTS_DIR)
Qwen_RAG_blind_answers = load_jsonl("hit_k_1_-all-blind-Qwen3-0.6B.jsonl", OUTPUTS_DIR) 

Qwen_Oracle_test_answers = load_jsonl("hit_k_1_-all-test-Qwen3-0.6B-oracle.jsonl", NO_REQUESTED_OUTPUTS_DIR)
Qwen_Oracle_blind_answers = load_jsonl("hit_k_1_-all-blind-Qwen3-0.6B-oracle.jsonl", NO_REQUESTED_OUTPUTS_DIR)

Loading Llama results...
✅ File loaded successfully: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\no_requested_outputs\hit_k_1_-all-test-Llama-3.2-3B-Instruct-baseline.jsonl (2000 query)
✅ File loaded successfully: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\no_requested_outputs\hit_k_1_-all-blind-Llama-3.2-3B-Instruct-baseline.jsonl (1322 query)
✅ File loaded successfully: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\outputs\hit_k_1_-all-test-Llama-3.2-3B-Instruct.jsonl (2000 query)
✅ File loaded successfully: C:/Users/djmat/Desktop/Magistrale - Engineering in CS and AI - Sapienza/Multilingual_Natural_Language_Processing/MNLP_Project_2026\outputs\hit_k_1_-all-blind-Llama-3.2-3B-Instruct.jsonl (1322 query)
✅ File loaded successfull

## **Standard metrics**

In [33]:
# Download required NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)

# Clean and normalize text
def normalize_answer(s):

    # Convert text to lowercase
    def lower(text):
        return text.lower()

    # Remove punctuation
    def remove_punc(text):
        return ''.join(ch for ch in text if ch not in set(string.punctuation))

    # Remove articles 
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    # Remove extra whitespace (tabs, newlines, multiple spaces)
    def white_space_fix(text):
        return ' '.join(text.split())

    if not s:
        return ""
    return white_space_fix(remove_articles(remove_punc(lower(str(s)))))


# Calculate Exact Match (EM)
def compute_em(generated, ground_truth):
    return int(normalize_answer(generated) == normalize_answer(ground_truth))

# Calculate sub-string match (subEM)
def compute_sub_em(generated, ground_truth):
    normalized_ground_truth = normalize_answer(ground_truth)
    return int(bool(normalized_ground_truth) and normalized_ground_truth in normalize_answer(generated))

# Calculate METEOR score
def compute_meteor(generated, ground_truth):
    reference_tokens = word_tokenize(normalize_answer(ground_truth))
    hypothesis_tokens = word_tokenize(normalize_answer(generated))
    if not reference_tokens or not hypothesis_tokens:
        return 0.0
    return meteor_score.single_meteor_score(reference_tokens, hypothesis_tokens)


# Computing metrics for a given set of predictions and the corresponding dataset
def evaluate_metrics(predictions_dict, dataset):

    # Prepare the correct answers dictionary
    ground_truths = {item['query_id']: item['short_answer'] for item in dataset}

    em_count = 0
    sub_em_count= 0
    total_meteor= 0.0
    total = len(dataset)

    for pred in predictions_dict.values():
        generated = pred.get('generated_answer', '')
        truth = ground_truths.get(pred['query_id'], '')

        em_count += compute_em(generated, truth)
        sub_em_count += compute_sub_em(generated, truth)
        total_meteor += compute_meteor(generated, truth)

    results = {
        "EM (%)": (em_count / total) * 100,
        "subEM (%)": (sub_em_count / total) * 100,
        "METEOR": total_meteor / total,
    }

    return results

# Collect results and build the final DataFrame
summary_data = [
    {"Model": "Llama-3.2-3B", "Setup": "Baseline", **evaluate_metrics(Llama_baseline_test_answers, test)},
    {"Model": "Llama-3.2-3B", "Setup": "RAG",      **evaluate_metrics(Llama_RAG_test_answers, test)},
    {"Model": "Llama-3.2-3B", "Setup": "Oracle",   **evaluate_metrics(Llama_Oracle_test_answers, test)},

    {"Model": "Qwen3-0.6B",   "Setup": "Baseline", **evaluate_metrics(Qwen_baseline_test_answers, test)},
    {"Model": "Qwen3-0.6B",   "Setup": "RAG",      **evaluate_metrics(Qwen_RAG_test_answers, test)},
    {"Model": "Qwen3-0.6B",   "Setup": "Oracle",   **evaluate_metrics(Qwen_Oracle_test_answers, test)}
]

# Format and display the table
df_results = pd.DataFrame(summary_data)
df_results = df_results.round({"EM (%)": 4, "subEM (%)": 4, "METEOR": 4})

display(df_results)

,Model,Setup,EM (%),subEM (%),METEOR
0,Llama-3.2-3B,Baseline,0.0,23.95,0.1403
1,Llama-3.2-3B,RAG,0.0,40.70,0.2474
2,Llama-3.2-3B,Oracle,0.0,45.00,0.2669
3,Qwen3-0.6B,Baseline,0.0,3.30,0.0384
4,Qwen3-0.6B,RAG,0.0,30.15,0.1854
5,Qwen3-0.6B,Oracle,0.0,33.20,0.2007


## **LLM as a Judge**

In [ ]:
llama_answers = dict(list(Llama_RAG_test_answers.items())[:500])
qwen_answers = dict(list(Qwen_RAG_test_answers.items())[:500])

JUDGE_PROMPT = """\
You are an evaluation judge. Answer ONLY with 0 or 1.

Score 1 – the correct answer is present in any valid form
          (e.g. "twenty-four" matches "24"; "753 BCE" matches "Rome was founded in 753 BCE").
Score 0 – the correct answer is absent or wrong
          (e.g. "Rome" when the correct answer is "Milan").

Question: {query}
Correct answer: {short_answer}
Generated answer: {generated_answer}

Result:"""

# This function takes the raw output from the judge and tries to parse it into a score (0 or 1) and a validity flag (True if it's a valid score, False otherwise)
def parse_score(raw: str) -> tuple[int, bool]:

    first_char = raw.strip()[:1]
    if first_char == "1":
        return 1, True
    if first_char == "0":
        return 0, True
    return 0, False         

def llm_as_a_judge(lm_answers, dataset):

    results = []
    invalid_count = 0

    print("⚖️  Starting LLM-as-a-Judge evaluation...")

    for q_id, data in lm_answers:

        prompt = JUDGE_PROMPT.format(
            query= data['query'],
            short_answer=dataset[dataset['query_id'].index(q_id)]['short_answer'],
            generated_answer=data['generated_answer'],
        )

        # Set the batch size dynamically: 8 on Colab, 16 locally
        current_batch_size = 8 if colab else 16

        outputs = llama(
            prompts,
            batch_size=current_batch_size,
            max_new_tokens=50,
            truncation=True,
            return_full_text=False,
            do_sample=False,
            generation_config=None,
            pad_token_id=generator.tokenizer.eos_token_id
        )

        raw   = outputs[0]['generated_text'].strip()
        score, valid = parse_score(raw)

        if not valid:
            invalid_count += 1

        results.append({
            "query_id":         q_id,
            "query":            data['query'],
            "short_answer":     data['short_answer'],
            "generated_answer": data['generated_answer'],
            "judge_raw":        raw,     # utile per debug
            "judge_score":      score,
            "valid_response":   valid,
        })
    return results, invalid_count

# Risultati
df_results = pd.DataFrame(results)

accuracy = df_results['judge_score'].mean() * 100
print(f"\n🏆 Accuratezza LLM-as-a-Judge : {accuracy:.2f}%")
print(f"⚠️  Risposte non valide (né 0 né 1): {invalid_count}/{len(df_results)}")

# Salva per analisi successive
df_results.to_csv("judge_results.csv", index=False)
print("💾 Risultati salvati in judge_results.csv")

# Distribuzione rapida dei punteggi
print("\nDistribuzione punteggi:")
print(df_results['judge_score'].value_counts().to_string())